# Exploratory Data Analysis (EDA)This notebook performs descriptive statistics, quality control, and exploratory visualization before the preregistered statistical analyses.

In [ ]:
# Import the core analysis libraries used throughout the notebook.from pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdimport seaborn as sns# Configure a clean plotting theme and publication-sized defaults.sns.set_theme(style="whitegrid", context="talk")plt.rcParams.update(    {        "figure.dpi": 300,        "savefig.dpi": 300,        "axes.titlesize": 16,        "axes.labelsize": 14,        "xtick.labelsize": 12,        "ytick.labelsize": 12,        "legend.fontsize": 11,    })# Resolve the repository root from either the repo root or the notebooks folder.repo_root = Path.cwd().resolve()if not (repo_root / "data" / "derived").exists():    repo_root = repo_root.parentassert (repo_root / "data" / "derived").exists(), (    "Run this notebook from the repository root or from the notebooks directory.")# Define the output directories for notebook-specific tables and figures.derived_dir = repo_root / "data" / "derived"figure_dir = repo_root / "figures" / "notebook_eda"table_dir = repo_root / "tables" / "notebook_eda"figure_dir.mkdir(parents=True, exist_ok=True)table_dir.mkdir(parents=True, exist_ok=True)# Use a consistent speaker order and palette in every relevant figure.speaker_order = ["adult_female", "adult_male", "other_child"]speaker_palette = {    "adult_female": "#1b9e77",    "adult_male": "#d95f02",    "other_child": "#7570b3",}# Define a helper that saves every figure as both PNG and PDF.def save_figure(fig: plt.Figure, stem: str) -> None:    fig.savefig(figure_dir / f"{stem}.png", bbox_inches="tight")    fig.savefig(figure_dir / f"{stem}.pdf", bbox_inches="tight")print(f"Repository root: {repo_root}")print(f"Figure output directory: {figure_dir}")print(f"Table output directory: {table_dir}")

## Load DatasetsThis section reads the already-created derived datasets and checks that only public participant identifiers are present.

In [ ]:
# Read the derived datasets created by the package pipeline.final_master = pd.read_csv(derived_dir / "final_master.csv")input_long = pd.read_csv(derived_dir / "input_long.csv")input_output_long = pd.read_csv(derived_dir / "input_output_long.csv")# Store the datasets in one mapping so the same validation can be applied to each file.datasets = {    "final_master": final_master,    "input_long": input_long,    "input_output_long": input_output_long,}# Check that public participant identifiers are present and private identifiers are absent.for name, dataframe in datasets.items():    assert "participant_id" in dataframe.columns, f"participant_id missing from {name}"    assert "original_par_id" not in dataframe.columns, (        f"original_par_id should not appear in public dataset {name}"    )    print(f"\n{name} shape: {dataframe.shape}")    print(f"{name} columns:")    print(list(dataframe.columns))    print(f"{name} first five rows:")    print(dataframe.head().to_string(index=False))

## Section 1. Dataset OverviewThis section summarizes the participant sample and saves descriptive tables for the main demographic variables.

In [ ]:
# Standardize child sex labels for simple female and male counts.sex_series = final_master["child_sex"].astype(str).str.strip().str.upper()# Build a compact overview table for the participant sample and recording duration.overview_summary = pd.DataFrame(    [        {"metric": "total_participants", "value": int(final_master["participant_id"].nunique())},        {"metric": "number_females", "value": int(sex_series.eq("F").sum())},        {"metric": "number_males", "value": int(sex_series.eq("M").sum())},        {"metric": "age_mean", "value": final_master["age_months"].mean()},        {"metric": "age_sd", "value": final_master["age_months"].std()},        {"metric": "age_min", "value": final_master["age_months"].min()},        {"metric": "age_max", "value": final_master["age_months"].max()},        {"metric": "recording_duration_mean", "value": final_master["recording_duration_hours"].mean()},        {"metric": "recording_duration_sd", "value": final_master["recording_duration_hours"].std()},        {"metric": "recording_duration_min", "value": final_master["recording_duration_hours"].min()},        {"metric": "recording_duration_max", "value": final_master["recording_duration_hours"].max()},    ])overview_summary.to_csv(table_dir / "dataset_overview_summary.csv", index=False)# Create frequency tables for the main background variables.location_frequency = final_master["Location"].value_counts(dropna=False).rename_axis("Location").reset_index(name="count")mother_education_frequency = final_master["mother_education"].value_counts(dropna=False).rename_axis("mother_education").reset_index(name="count")father_education_frequency = final_master["father_education"].value_counts(dropna=False).rename_axis("father_education").reset_index(name="count")location_frequency.to_csv(table_dir / "location_frequency.csv", index=False)mother_education_frequency.to_csv(table_dir / "mother_education_frequency.csv", index=False)father_education_frequency.to_csv(table_dir / "father_education_frequency.csv", index=False)print("Dataset overview summary:")print(overview_summary.to_string(index=False))print("\nLocation frequency:")print(location_frequency.to_string(index=False))print("\nMother education frequency:")print(mother_education_frequency.to_string(index=False))print("\nFather education frequency:")print(father_education_frequency.to_string(index=False))

## Section 2. Missing DataThis section reports variable-level missingness in the public participant-level dataset.

In [ ]:
# Count missing values and convert them into percentages for every variable.missing_summary = pd.DataFrame(    {        "variable": final_master.columns,        "missing_count": final_master.isna().sum().values,        "missing_percentage": (final_master.isna().mean().values * 100),    })missing_summary = missing_summary.sort_values(["missing_count", "variable"], ascending=[False, True], kind="stable").reset_index(drop=True)missing_summary.to_csv(table_dir / "final_master_missing_data.csv", index=False)print(missing_summary.to_string(index=False))

## Section 3. Recording Duration Quality ControlThese figures summarize recording duration and its relationship with age.

In [ ]:
# Summarize recording duration before plotting the quality-control figures.recording_duration_summary = final_master["recording_duration_hours"].describe()recording_duration_summary.to_frame(name="recording_duration_hours").to_csv(    table_dir / "recording_duration_descriptives.csv")print(recording_duration_summary.to_string())# Figure 1: Plot the recording duration distribution as a histogram.fig, ax = plt.subplots(figsize=(8, 5))sns.histplot(final_master, x="recording_duration_hours", bins=20, color="#4c78a8", ax=ax)ax.set_title("Figure 1. Recording Duration Distribution")ax.set_xlabel("Recording duration (hours)")ax.set_ylabel("Number of participants")fig.tight_layout()save_figure(fig, "figure_1_recording_duration_histogram")plt.show()# Figure 2: Visualize the recording duration spread with a boxplot.fig, ax = plt.subplots(figsize=(6, 4))sns.boxplot(y=final_master["recording_duration_hours"], color="#72b7b2", ax=ax)ax.set_title("Figure 2. Recording Duration Boxplot")ax.set_xlabel("")ax.set_ylabel("Recording duration (hours)")fig.tight_layout()save_figure(fig, "figure_2_recording_duration_boxplot")plt.show()# Figure 3: Relate recording duration to age with a regression overlay.fig, ax = plt.subplots(figsize=(7, 5))sns.scatterplot(    data=final_master,    x="age_months",    y="recording_duration_hours",    color="#4c78a8",    s=70,    alpha=0.8,    ax=ax,)sns.regplot(    data=final_master,    x="age_months",    y="recording_duration_hours",    scatter=False,    color="#222222",    ax=ax,)ax.set_title("Figure 3. Recording Duration by Age")ax.set_xlabel("Age (months)")ax.set_ylabel("Recording duration (hours)")fig.tight_layout()save_figure(fig, "figure_3_recording_duration_by_age")plt.show()

## Section 4. Age DistributionThis section visualizes the age distribution and reports core descriptive statistics.

In [ ]:
# Summarize the age distribution numerically before plotting it.age_summary = pd.DataFrame(    [        {"metric": "mean", "value": final_master["age_months"].mean()},        {"metric": "median", "value": final_master["age_months"].median()},        {"metric": "sd", "value": final_master["age_months"].std()},        {"metric": "min", "value": final_master["age_months"].min()},        {"metric": "max", "value": final_master["age_months"].max()},    ])age_summary.to_csv(table_dir / "age_distribution_summary.csv", index=False)print(age_summary.to_string(index=False))# Figure 4: Display the age distribution with a histogram and KDE.fig, ax = plt.subplots(figsize=(8, 5))sns.histplot(final_master["age_months"], kde=True, bins=20, color="#f28e2b", ax=ax)ax.axvline(final_master["age_months"].mean(), color="#222222", linestyle="--", label="Mean")ax.axvline(final_master["age_months"].median(), color="#666666", linestyle=":", label="Median")ax.set_title("Figure 4. Age Distribution")ax.set_xlabel("Age (months)")ax.set_ylabel("Number of participants")ax.legend()fig.tight_layout()save_figure(fig, "figure_4_age_distribution")plt.show()

## Section 5. Vocal Input by SpeakerThis section compares hourly vocal input across adult female, adult male, and other child speakers.

In [ ]:
# Create a notebook-specific copy and derive minutes per hour from the duration rate.input_long_eda = input_long.copy()input_long_eda["speaker"] = pd.Categorical(input_long_eda["speaker"], categories=speaker_order, ordered=True)input_long_eda["input_duration_min_hour"] = input_long_eda["input_duration_hour"] / 60.0# Figure 5: Compare count-per-hour distributions across speaker types.fig, ax = plt.subplots(figsize=(8, 5))sns.boxplot(data=input_long_eda, x="speaker", y="input_count_hour", order=speaker_order, palette=speaker_palette, ax=ax)sns.stripplot(data=input_long_eda, x="speaker", y="input_count_hour", order=speaker_order, color="#222222", alpha=0.55, size=4, ax=ax)ax.set_title("Figure 5. Input Count per Hour by Speaker")ax.set_xlabel("Speaker")ax.set_ylabel("Input count per hour")fig.tight_layout()save_figure(fig, "figure_5_input_count_hour_by_speaker")plt.show()# Figure 6: Compare duration-per-hour distributions in minutes across speaker types.fig, ax = plt.subplots(figsize=(8, 5))sns.boxplot(data=input_long_eda, x="speaker", y="input_duration_min_hour", order=speaker_order, palette=speaker_palette, ax=ax)sns.stripplot(data=input_long_eda, x="speaker", y="input_duration_min_hour", order=speaker_order, color="#222222", alpha=0.55, size=4, ax=ax)ax.set_title("Figure 6. Input Duration per Hour by Speaker")ax.set_xlabel("Speaker")ax.set_ylabel("Input duration (minutes per hour)")fig.tight_layout()save_figure(fig, "figure_6_input_duration_min_hour_by_speaker")plt.show()# Summarize the speaker-specific distributions with central tendency and spread statistics.speaker_summary = input_long_eda.groupby("speaker", observed=True).agg(    input_count_hour_mean=("input_count_hour", "mean"),    input_count_hour_sd=("input_count_hour", "std"),    input_count_hour_median=("input_count_hour", "median"),    input_count_hour_q1=("input_count_hour", lambda x: x.quantile(0.25)),    input_count_hour_q3=("input_count_hour", lambda x: x.quantile(0.75)),    input_count_hour_min=("input_count_hour", "min"),    input_count_hour_max=("input_count_hour", "max"),    input_duration_min_hour_mean=("input_duration_min_hour", "mean"),    input_duration_min_hour_sd=("input_duration_min_hour", "std"),    input_duration_min_hour_median=("input_duration_min_hour", "median"),    input_duration_min_hour_q1=("input_duration_min_hour", lambda x: x.quantile(0.25)),    input_duration_min_hour_q3=("input_duration_min_hour", lambda x: x.quantile(0.75)),    input_duration_min_hour_min=("input_duration_min_hour", "min"),    input_duration_min_hour_max=("input_duration_min_hour", "max"),)speaker_summary = speaker_summary.reset_index()speaker_summary["input_count_hour_iqr"] = speaker_summary["input_count_hour_q3"] - speaker_summary["input_count_hour_q1"]speaker_summary["input_duration_min_hour_iqr"] = speaker_summary["input_duration_min_hour_q3"] - speaker_summary["input_duration_min_hour_q1"]speaker_summary.to_csv(table_dir / "speaker_input_summary.csv", index=False)print(speaker_summary.to_string(index=False))

## Section 6. Input CompositionThis section expresses each participant's input as the percentage of counts contributed by each speaker class.

In [ ]:
# Pivot the long-format counts into a participant-by-speaker matrix.composition_counts = input_long_eda.pivot(index="participant_id", columns="speaker", values="input_count_hour")composition_counts = composition_counts.reindex(columns=speaker_order)# Convert the hourly counts into within-participant percentages of total input.composition_pct = composition_counts.div(composition_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0) * 100.0composition_pct = composition_pct.merge(final_master[["participant_id", "age_months"]], left_index=True, right_on="participant_id", how="left")composition_pct = composition_pct.sort_values(["age_months", "participant_id"], kind="stable").reset_index(drop=True)composition_pct.to_csv(table_dir / "input_composition_percentages.csv", index=False)# Figure 7: Plot speaker composition as stacked percentages ordered by age.fig, ax = plt.subplots(figsize=(max(10, len(composition_pct) * 0.3), 6))bottom = np.zeros(len(composition_pct))for speaker in speaker_order:    values = composition_pct[speaker].to_numpy()    ax.bar(composition_pct["participant_id"], values, bottom=bottom, label=speaker, color=speaker_palette[speaker])    bottom += valuesax.set_title("Figure 7. Input Composition by Participant")ax.set_xlabel("Participant ID (ordered by age)")ax.set_ylabel("Input share (%)")ax.set_ylim(0, 100)ax.tick_params(axis="x", rotation=90)ax.legend(title="Speaker")fig.tight_layout()save_figure(fig, "figure_7_input_composition_stacked_bar")plt.show()

## Section 7. Input Changes with AgeThese plots relate participant age to hourly input counts and durations, while using recording duration to scale point size.

In [ ]:
# Figure 8: Show the age relationship for hourly input counts by speaker.fig, ax = plt.subplots(figsize=(8, 5))sns.scatterplot(    data=input_long_eda,    x="age_months",    y="input_count_hour",    hue="speaker",    hue_order=speaker_order,    palette=speaker_palette,    size="recording_duration_hours",    sizes=(40, 220),    alpha=0.8,    ax=ax,)for speaker in speaker_order:    subset = input_long_eda.loc[input_long_eda["speaker"] == speaker]    sns.regplot(data=subset, x="age_months", y="input_count_hour", scatter=False, color=speaker_palette[speaker], ax=ax)ax.set_title("Figure 8. Input Count per Hour by Age")ax.set_xlabel("Age (months)")ax.set_ylabel("Input count per hour")fig.tight_layout()save_figure(fig, "figure_8_input_count_hour_by_age")plt.show()# Figure 9: Repeat the age relationship using input duration in minutes per hour.fig, ax = plt.subplots(figsize=(8, 5))sns.scatterplot(    data=input_long_eda,    x="age_months",    y="input_duration_min_hour",    hue="speaker",    hue_order=speaker_order,    palette=speaker_palette,    size="recording_duration_hours",    sizes=(40, 220),    alpha=0.8,    ax=ax,)for speaker in speaker_order:    subset = input_long_eda.loc[input_long_eda["speaker"] == speaker]    sns.regplot(data=subset, x="age_months", y="input_duration_min_hour", scatter=False, color=speaker_palette[speaker], ax=ax)ax.set_title("Figure 9. Input Duration per Hour by Age")ax.set_xlabel("Age (months)")ax.set_ylabel("Input duration (minutes per hour)")fig.tight_layout()save_figure(fig, "figure_9_input_duration_min_hour_by_age")plt.show()

## Section 8. Key Child OutputThese figures focus on the child's own vocal output over age.

In [ ]:
# Derive key child duration in minutes per hour for a more interpretable plot scale.final_master_eda = final_master.copy()final_master_eda["key_child_duration_min_hour"] = final_master_eda["key_child_duration_hour"] / 60.0# Figure 10: Plot key child count rate as a function of age.fig, ax = plt.subplots(figsize=(8, 5))sns.scatterplot(    data=final_master_eda,    x="age_months",    y="key_child_count_hour",    size="recording_duration_hours",    sizes=(40, 220),    color="#4c78a8",    alpha=0.8,    ax=ax,)sns.regplot(data=final_master_eda, x="age_months", y="key_child_count_hour", scatter=False, color="#222222", ax=ax)ax.set_title("Figure 10. Key Child Count per Hour by Age")ax.set_xlabel("Age (months)")ax.set_ylabel("Key child count per hour")fig.tight_layout()save_figure(fig, "figure_10_key_child_count_hour_by_age")plt.show()# Figure 11: Repeat the plot using key child duration in minutes per hour.fig, ax = plt.subplots(figsize=(8, 5))sns.scatterplot(    data=final_master_eda,    x="age_months",    y="key_child_duration_min_hour",    size="recording_duration_hours",    sizes=(40, 220),    color="#f28e2b",    alpha=0.8,    ax=ax,)sns.regplot(data=final_master_eda, x="age_months", y="key_child_duration_min_hour", scatter=False, color="#222222", ax=ax)ax.set_title("Figure 11. Key Child Duration per Hour by Age")ax.set_xlabel("Age (months)")ax.set_ylabel("Key child duration (minutes per hour)")fig.tight_layout()save_figure(fig, "figure_11_key_child_duration_min_hour_by_age")plt.show()

## Section 9. Input-Output RelationshipThese faceted plots compare caregiver and other-child input with the child's output.

In [ ]:
# Prepare duration variables in minutes per hour for the faceted input-output plots.input_output_eda = input_output_long.copy()input_output_eda["speaker"] = pd.Categorical(input_output_eda["speaker"], categories=speaker_order, ordered=True)input_output_eda["input_duration_min_hour"] = input_output_eda["input_duration_hour"] / 60.0input_output_eda["key_child_duration_min_hour"] = input_output_eda["key_child_duration_hour"] / 60.0# Figure 12: Compare hourly input counts with key child output counts in separate speaker panels.fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)for axis, speaker in zip(axes, speaker_order):    subset = input_output_eda.loc[input_output_eda["speaker"] == speaker]    sns.scatterplot(        data=subset,        x="input_count_hour",        y="key_child_count_hour",        size="recording_duration_hours",        sizes=(40, 220),        color=speaker_palette[speaker],        alpha=0.8,        legend=False,        ax=axis,    )    sns.regplot(data=subset, x="input_count_hour", y="key_child_count_hour", scatter=False, color="#222222", ax=axis)    axis.set_title(speaker)    axis.set_xlabel("Input count per hour")axes[0].set_ylabel("Key child count per hour")fig.suptitle("Figure 12. Input Count and Key Child Count Relationship", y=1.02)fig.tight_layout()save_figure(fig, "figure_12_input_output_count_relationship")plt.show()# Figure 13: Repeat the speaker-faceted comparison using duration in minutes per hour.fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)for axis, speaker in zip(axes, speaker_order):    subset = input_output_eda.loc[input_output_eda["speaker"] == speaker]    sns.scatterplot(        data=subset,        x="input_duration_min_hour",        y="key_child_duration_min_hour",        size="recording_duration_hours",        sizes=(40, 220),        color=speaker_palette[speaker],        alpha=0.8,        legend=False,        ax=axis,    )    sns.regplot(data=subset, x="input_duration_min_hour", y="key_child_duration_min_hour", scatter=False, color="#222222", ax=axis)    axis.set_title(speaker)    axis.set_xlabel("Input duration (minutes per hour)")axes[0].set_ylabel("Key child duration (minutes per hour)")fig.suptitle("Figure 13. Input Duration and Key Child Duration Relationship", y=1.02)fig.tight_layout()save_figure(fig, "figure_13_input_output_duration_relationship")plt.show()

## Section 10. Correlation MatrixThis section visualizes Pearson correlations among the main age, duration, input, and output measures.

In [ ]:
# Select the main numeric variables requested for the correlation matrix.correlation_columns = [    "age_months",    "recording_duration_hours",    "adult_female_count_hour",    "adult_male_count_hour",    "other_child_count_hour",    "key_child_count_hour",    "adult_female_duration_hour",    "adult_male_duration_hour",    "other_child_duration_hour",    "key_child_duration_hour",]correlation_matrix = final_master_eda[correlation_columns].corr(method="pearson")correlation_matrix.to_csv(table_dir / "correlation_matrix.csv")# Plot the Pearson correlation matrix as a heatmap.fig, ax = plt.subplots(figsize=(10, 8))sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="vlag", square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)ax.set_title("Correlation Matrix of Main Study Variables")fig.tight_layout()save_figure(fig, "figure_14_correlation_matrix")plt.show()

## Section 11. Outlier InspectionThis section reports the highest and lowest observed values for key descriptive variables without removing any participants.

In [ ]:
# Collect the requested top-five participant tables for the main descriptive variables.top_metrics = [    "recording_duration_hours",    "key_child_count_hour",    "adult_female_count_hour",    "adult_male_count_hour",    "other_child_count_hour",]top_outlier_tables = []for metric in top_metrics:    table = final_master_eda[["participant_id", metric]].sort_values(metric, ascending=False, kind="stable").head(5).copy()    table.insert(0, "metric", metric)    top_outlier_tables.append(table)top_outliers = pd.concat(top_outlier_tables, ignore_index=True)top_outliers.to_csv(table_dir / "top_outlier_participants.csv", index=False)# Also report the lowest five recording durations as a separate quality-control table.bottom_recording_duration = final_master_eda[["participant_id", "recording_duration_hours"]].sort_values("recording_duration_hours", ascending=True, kind="stable").head(5).copy()bottom_recording_duration.to_csv(table_dir / "bottom_recording_duration_participants.csv", index=False)print("Top outlier tables:")print(top_outliers.to_string(index=False))print("\nBottom five recording durations:")print(bottom_recording_duration.to_string(index=False))